# 📊 V56: DEEP HYDROLOGICAL EDA (The Grand Audit)\n\nBerdasarkan permintaan investigasi dari *Data Scientist* senior, *notebook* ini dirancang khusus untuk membedah anatomi murni dari sungai Bengawan Solo secara statistik.\n\nKita akan mencari tahu stasiun mana yang menjadi 'Monster' penghancur RMSE, mencari durasi rambatan air yang paling akurat secara matematis (CCF), dan mengukur seberapa ekstrem lonjakan (*outlier*) air yang sesungguhnya!

In [ ]:
!pip install -q statsmodels
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import ccf, acf, pacf

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

# Deteksi Environment
if not 'path' in locals():
    if os.path.exists('/kaggle/input'):
        dataset_folders = [f for f in os.listdir('/kaggle/input') if os.path.isdir(f'/kaggle/input/{f}')]
        path = f'/kaggle/input/{dataset_folders[0]}/' if dataset_folders else '/kaggle/input/'
    elif os.path.exists('/content'):
        from google.colab import drive
        drive.mount('/content/drive')
        path = '/content/drive/MyDrive/SSF-SSDS/'
    else:
        path = './'

train_df = pd.read_csv(path + 'train.csv', parse_dates=['datetime'])
lingkungan_df = pd.read_csv(path + 'data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
print("Dataset berhasil dimuat!")

## 1. The Variance Monster (Distribusi & Skala TMA)\nKita akan melihat secara visual mana stasiun yang memiliki fluktuasi target paling ekstrem. Mengingat RMSE Kaggle dihitung secara **GLOBAL** (*baris per baris*), stasiun dengan rentang (scale) yang lebar akan mendominasi 90% dari total *error* jika tebakannya meleset sedikit saja.

In [ ]:
plt.figure(figsize=(20, 10))
sns.boxplot(x='nama_pos', y='tma_mdpl', data=train_df, palette='viridis')
plt.xticks(rotation=90)
plt.title('Distribusi TMA (mdpl) per Stasiun: Mencari "The Variance Monster"', fontsize=16, fontweight='bold')
plt.ylabel('TMA (Meter Diatas Permukaan Laut)')
plt.tight_layout()
plt.show()

# Tampilkan statistik numerik stasiun dengan variansi terbesar
std_df = train_df.groupby('nama_pos')['tma_mdpl'].std().sort_values(ascending=False).reset_index()
std_df.columns = ['Stasiun', 'Standar Deviasi (Meter)']
print("\nTop 5 Stasiun dengan Fluktuasi Paling Mematikan (Prioritas Penyelamatan):")
display(std_df.head())

## 2. Profiling Time-Series per Stasiun\nApakah datanya memiliki pola musiman (mudah), atau fluktuasi acak yang sangat tajam (*flash flood*)? Kita buat *subplot* untuk 30 stasiun.

In [ ]:
stations = sorted(train_df['nama_pos'].unique())

fig, axes = plt.subplots(10, 3, figsize=(24, 40))
axes = axes.flatten()

for i, st in enumerate(stations):
    st_data = train_df[train_df['nama_pos'] == st]
    axes[i].plot(st_data['datetime'], st_data['tma_mdpl'], color='navy', alpha=0.8)
    axes[i].set_title(st, fontweight='bold')
    axes[i].set_ylabel('TMA')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Heatmap Ketersediaan Data (Missing Values)\nAnalisis ini penting untuk melihat apakah ada stasiun yang sensornya mati total berbulan-bulan. Sensor yang mati dapat mengacaukan perhitungan fitur *Moving Average/EWM* kita.

In [ ]:
# Pivot untuk melihat missing data secara kronologis
train_pivot = train_df.pivot(index='datetime', columns='nama_pos', values='tma_mdpl')

plt.figure(figsize=(20, 10))
sns.heatmap(train_pivot.isnull(), cbar=False, cmap='magma')
plt.title('Heatmap Missing Values TMA (Hitam = Ada Data, Kuning Terang = Sensor Mati/Kosong)', fontsize=14, fontweight='bold')
plt.ylabel('Timeline (Jan 2023 - Sep 2025)')
plt.show()

missing_pct = train_pivot.isnull().mean() * 100
print("\nTop 5 Stasiun dengan Data Paling Banyak Bolong (%):")
display(missing_pct.sort_values(ascending=False).head())

## 4. Cross-Correlation (CCF): Mengukur Waktu Tempuh Matematis\nIni adalah inti dari investigasi hari ini. Alih-alih menebak *delay* air dari hulu, kita akan mencari secara pasti berapa JAM *delay* optimal (korelasi tertinggi) antara hujan (Rainfall) di suatu stasiun dengan peningkatan tinggi air (TMA) di stasiun tersebut/hilirnya.

In [ ]:
# Merge TMA dan Hujan di level stasiun yang sama dulu
df_merged = pd.merge(train_df, lingkungan_df[['datetime', 'nama_pos', 'rainfall_mm']], on=['datetime', 'nama_pos'], how='left')

plt.figure(figsize=(24, 30))

for i, st in enumerate(stations):
    st_data = df_merged[df_merged['nama_pos'] == st].dropna()
    
    if len(st_data) > 100:
        # Cross Correlation antara Rainfall (X) dan TMA (Y)
        # Menjawab: Hujan hari ini, efeknya ke TMA berapa jam lagi?
        ccf_vals = ccf(st_data['rainfall_mm'], st_data['tma_mdpl'])[:120] # Cek delay sampai 120 jam (5 hari)
        
        plt.subplot(10, 3, i+1)
        plt.plot(ccf_vals, color='crimson')
        
        # Cari lag dengan korelasi maksimal
        max_lag = np.argmax(ccf_vals)
        max_corr = ccf_vals[max_lag]
        
        plt.axvline(x=max_lag, color='blue', linestyle='--')
        plt.title(f"{st}\nMax Corr: {max_corr:.3f} at Lag {max_lag} jam", fontsize=10)
        plt.xlabel("Delay Hujan (Jam)")
        plt.ylabel("Korelasi CCF")

plt.tight_layout()
plt.show()

## 5. Autocorrelation (ACF) - Inertia Sungai\nBerapa lama sungai menyimpan air sebelum surut? Jika nilai ACF masih sangat tinggi di *lag* yang jauh (misal 120 jam), berarti sungai itu lambat surut (bertindak seperti reservoir raksasa).

In [ ]:
plt.figure(figsize=(24, 30))

for i, st in enumerate(stations):
    st_data = train_df[train_df['nama_pos'] == st]['tma_mdpl'].dropna()
    
    if len(st_data) > 100:
        acf_vals = acf(st_data, nlags=120)
        
        plt.subplot(10, 3, i+1)
        plt.plot(acf_vals, color='darkgreen')
        plt.axhline(y=0.5, color='gray', linestyle='--')
        plt.title(f"{st} (ACF)", fontsize=10)
        plt.xlabel("Lag (Jam)")
        plt.ylabel("Autocorrelation")

plt.tight_layout()
plt.show()

## 6. Analisis Ekstrem & Clipping Outliers\nSatu baris data yang ekstrim (misal karena *glitch* sensor) akan di kuadratkan oleh rumus RMSE dan membumihanguskan skor kita. Mari kita lihat *Percentile* 1% dan 99%.

In [ ]:
plt.figure(figsize=(20, 10))
for i, st in enumerate(['Napel', 'Bojonegoro - Kali Kethek', 'Karangnongko']): # 3 Stasiun Sample
    st_data = train_df[train_df['nama_pos'] == st]['tma_mdpl'].dropna()
    
    p1 = np.percentile(st_data, 1)
    p99 = np.percentile(st_data, 99)
    
    plt.subplot(1, 3, i+1)
    sns.histplot(st_data, bins=50, kde=True, color='purple')
    plt.axvline(p1, color='red', linestyle='--', label=f'1% ({p1:.1f})')
    plt.axvline(p99, color='red', linestyle='--', label=f'99% ({p99:.1f})')
    plt.title(f'Distribusi Outlier: {st}')
    plt.legend()

plt.tight_layout()
plt.show()

## 7. Plot Dual-Axis: Respons TMA Terhadap Badai Hujan\nKita *zoom in* pada bulan Januari 2024 (Puncak Musim Hujan) untuk melihat secara mata telanjang seberapa sensitif grafik air mengikuti grafik hujan.

In [ ]:
sample_station = 'Bojonegoro - Kali Kethek'
df_sample = df_merged[(df_merged['nama_pos'] == sample_station) & 
                      (df_merged['datetime'] >= '2024-01-01') & 
                      (df_merged['datetime'] <= '2024-02-28')].copy()

fig, ax1 = plt.subplots(figsize=(16, 6))

color = 'tab:blue'
ax1.set_xlabel('Tanggal')
ax1.set_ylabel('TMA (mdpl)', color=color)
ax1.plot(df_sample['datetime'], df_sample['tma_mdpl'], color=color, linewidth=2, label='Tinggi Air')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  
color = 'tab:red'
ax2.set_ylabel('Curah Hujan (mm)', color=color)  
# Gunakan bar chart terbalik (diatas) agar seperti grafik curah hujan standar hidrologi
ax2.bar(df_sample['datetime'], df_sample['rainfall_mm'], color=color, alpha=0.3, width=0.1, label='Hujan')
ax2.tick_params(axis='y', labelcolor=color)

plt.title(f'Respons TMA Terhadap Hujan (Puncak Badai Jan-Feb 2024) - {sample_station}', fontsize=14, fontweight='bold')
fig.tight_layout()  
plt.show()